# Overview

This example uploads one scanning probe microscopy run to the platform: the run folder becomes a Sample Set with one Sample per measured position, a Measurement Set with one Measurement per Sample and its Setup, the run's records as files, and one hysteresis-loop Property per Sample.
A run folder is what the instrument exports — `summary.json` with the recipe, the session and one record per measured point, and `loops/` with the raw curves — and re-running the notebook adds only what is missing.

## Install the API client

The samples, measurements and files endpoints are not released yet, so the client is installed from its branch until it merges. Restart the kernel after this cell.

In [ ]:
%pip install -r requirements.txt

## Set Parameters

- **HOST**: platform the run is uploaded to
- **RUN_DIR**: the run folder beside this notebook — what the instrument exports, with `summary.json` and `loops/` inside it
- **PHYSICAL_ID**: the identifier written on the physical piece the measured positions are part of — every Sample carries it
- **ACCOUNT_SLUG**: account the data belongs to, empty for the default account
- **FILES**: which files to upload per measurement

In [ ]:
import urllib.parse

HOST = "https://alphafilm.mat3ra.com"
RUN_DIR = ""  # folder with summary.json, records etc
PHYSICAL_ID = ""  # the physical wafer's identifier (e.g. PDAC_COM5_01448)
FILES = ["records", "loops"]  # which folders to upload as files per measurement

url = urllib.parse.urlsplit(HOST)
address = {
    "host": url.hostname,
    "port": url.port or (443 if url.scheme == "https" else 80),
    "secure": url.scheme == "https",
}

## Authenticate and initialize API client

An API token from Preferences on `HOST`. It is typed in, not stored in the notebook; export
`ACCOUNT_ID` and `AUTH_TOKEN` before starting Jupyter to skip the prompts, or export
`OIDC_ACCESS_TOKEN` from a browser sign-in instead.


In [ ]:
import getpass
import os

# An API token from Preferences on the host above. Typed here, never stored in the notebook.
# Set ACCOUNT_ID and AUTH_TOKEN in the environment beforehand to skip the prompts.
if not os.environ.get("ACCOUNT_ID"):
    os.environ["ACCOUNT_ID"] = input("Account ID: ")
if not os.environ.get("AUTH_TOKEN"):
    os.environ["AUTH_TOKEN"] = getpass.getpass("API token: ")

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate(**address)

# Imports

In [ ]:
from pathlib import Path

from parse_utk import parse
from run_document import load, serialize
from upload_run import account_id, upload

## Parse the run folder

Read the run folder into the documents the platform stores. Nothing is uploaded yet.

In [ ]:
# reading the run folder and writing the run document: nothing here talks to the platform
parsed = parse(Path(RUN_DIR), PHYSICAL_ID)
document_path = serialize(parsed, "parsed")
run = load(document_path)
file_count = sum(len(files) for files in run["files"].values())
print(
    f"{run['physicalId']}: {len(run['samples'])} samples (ordered set) · run {run['run']}: "
    f"{len(run['measurements'])} measurements (ordered set, one per sample) · {len(run['set_files'])} set files "
    f"-> {file_count} measurement files · {len(run['properties'])} samples with a combined loop "
    f"· no curves: {len(run['skipped'])} samples"
)
print("run document:", document_path)

## Upload the run

Create the Sample Set and its Samples, the Measurement Set and one Measurement per Sample, the files and the loop Properties.

In [ ]:
upload(client, run, files=FILES)

## Find the run in the web app

The run is a folder in the account's Measurements tab, named after the run.

In [ ]:
print(f"Open {HOST}, your account's Measurements tab: {run['run']}")